In [ ]:
import pandas as pd
import numpy as np
import math
import text_utils
import heapq
from sklearn.metrics import confusion_matrix, classification_report
import json

In [ ]:
df = pd.read_csv("/Users/abhinavarora/Desktop/Android Spam Classifier/Machine Learning/Data Sets /final_balanced.csv")
print(df)


# v1 = label (spam/ham), v2 = text message
y = df['Class'].values  # labels
x = df['Text'].values  # raw text messages

rows = len(x)

# Shuffle and split
np.random.seed(42)  
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x[train_idx]  
x_test = x[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

In [ ]:
#A map that stores words in the format word:index
vocab_list = {}
#len(vocab_list)
vocab_size = 10000


#A simple tokenization of the text

#Creating tokens for training and tetsing which will later be vectorized
train_tokens = [text_utils.tokenize(x_train[i]) for i in range(len(x_train))]
test_tokens = [text_utils.tokenize(x_test[j]) for j in range(len(x_test))]

vocab_list = text_utils.build_vocab_list(vocab_size, x_train, train_tokens)
idf = text_utils.build_idf(x_train, train_tokens, vocab_list)


#Creating vectorized inputs for each training example in x_train
vectorized_inputs = [[0] * vocab_size for _ in range(len(x_train))]

vectorized_inputs = text_utils.vectorize(x_train, vocab_size, train_tokens, vocab_list, True, idf)

#Calculating the class ratios
class_ratio_map = text_utils.calculate_class_ratio(y_train)

In [ ]:
#This will store the phi parameter used in the Multinomial Distribution as (k, j) where k corresponds to the class and j corresponds to the feature
phi_dict = {}
#phi j,k = (occurences of a feature exisiting given a specific class)/(number of examples of that specific class)

#Calculates the word count from the vectorized inputs for each ham and spam classes
def calculate_word_count(vectorized_x, training_y):
    res_dict = {}
    for i in range(len(vectorized_x)):
        for j in range(len(vectorized_x[0])):
            if training_y[i] not in res_dict:
                res_dict[training_y[i]] = vectorized_x[i][j]
            else:
                res_dict[training_y[i]] += vectorized_x[i][j]
    
    return res_dict

def calculate_phi_dict(vectorized_x, training_y):
    feature_number = 0
    total_words = calculate_word_count(vectorized_x, training_y)
    for i in range(len(vectorized_x[0])):
        auxilary_dict = {element:0 for element in class_ratio_map}
        for j in range(len(vectorized_x)):
            auxilary_dict[training_y[j]] += vectorized_x[j][i]
    
        for elt in auxilary_dict:
            #Laplace smoothing to ensure log(0) probabilities never appear
            phi_dict[str(elt) + str(feature_number)] = (auxilary_dict[elt] + 1)/(total_words[elt] + vocab_size)
        
        feature_number += 1

calculate_phi_dict(vectorized_inputs, y_train)

n_train = len(y_train)

def predict_one(test_vector):
    heap = []
    heapq.heapify_max(heap)
    for element in class_ratio_map:
        s = 0
        for i in range(len(test_vector)):
            s += test_vector[i] * math.log(phi_dict[str(element) + str(i)])
        heapq.heappush_max(heap, (math.log(class_ratio_map[element]/n_train) + s, element))
    
    return heap[0][1]


#Vectorizing test inputs
vectorized_test_inputs = [[0] * 1000 for _ in range(len(x_test))]
vectorized_test_inputs = text_utils.vectorize(x_test, vocab_size, test_tokens, vocab_list, True, idf)

y_pred = []

#Making predictions
def make_all_predictions(test_x, test_y):
    accuracy = 0
    ham_count = 0
    spam_count = 0
    for i in range(len(test_x)):
        prediction = predict_one(test_x[i])
        y_pred.append(prediction)
        if(prediction == test_y[i]):
            accuracy += 1

        if prediction == 'ham':
            ham_count += 1
        else:
            spam_count += 1

    print(accuracy/len(test_x))

make_all_predictions(vectorized_test_inputs, y_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Model Performance (vocab_size=10000): Accuracy=94.8%, Spam Precision=0.93, Spam Recall=0.97, Ham Precision=0.97, Ham Recall=0.92, F1=0.95
    

In [ ]:
#Code to add parameters:
if False:
    parameter_dict = {
        "IDF": idf,
        "Phi dict": phi_dict,
        "Class ratios": class_ratio_map,
        "Train examples length": n_train,
        "Vocab size": vocab_size,
        "Vocab list": vocab_list
    }
    with open("model_params.json", "a") as params:
        json.dump(parameter_dict, params, indent=4)


In [ ]:
import requests

if False:
    messages = [
  "Hey, are we still meeting at 6 for dinner?",
  "URGENT: Your bank account has been temporarily suspended. Verify now.",
  "Your OTP for login is 482913. Do not share this code.",
  "Claim your free shopping voucher before midnight!",
  "Professor moved the quiz to Thursday.",
  "Congratulations! You have won a luxury vacation package.",
  "Amazon: Your package has been delivered.",
  "Final notice: unpaid toll detected. Pay immediately.",
  "Can you send me yesterday's lecture slides?",
  "Get instant loan approval with zero paperwork!",
  "Flight AI302 delayed by 45 minutes.",
  "Hot investment tip! Guaranteed 5x returns in crypto.",
  "Lunch at the student union?",
  "Your Netflix payment failed. Update billing details now.",
  "Reminder: dentist appointment tomorrow at 10 AM.",
  "Exclusive offer! Buy 1 get 3 free today only.",
  "The TA uploaded assignment solutions.",
  "Suspicious login detected. Confirm identity now.",
  "Running 10 mins late, start without me.",
  "Win ₹50,000 cash instantly by clicking here.",
  "Your Uber driver is arriving in 3 minutes.",
  "Account verification required to prevent closure.",
  "Movie night Friday?",
  "Work from home and earn ₹20,000/day guaranteed.",
  "Mom said to pick up bread on the way back.",
  "Tax refund pending. Submit details immediately.",
  "Your prescription is ready for pickup.",
  "Claim cashback reward before it expires.",
  "Bro your charger is in my room.",
  "Unlock premium content for free now.",
  "Can you review my resume tonight?",
  "Casino bonus unlocked! Deposit now.",
  "Meeting moved to 2 PM.",
  "Congratulations! Selected for an exclusive platinum card.",
  "Happy birthday! Hope you have a great one.",
  "Your account will be permanently deleted unless action is taken.",
  "The project deadline got extended.",
  "Crypto insiders are making huge profits. Join now.",
  "Package could not be delivered. Reschedule here.",
  "Can you call grandma this evening?",
  "The cafeteria pasta was surprisingly good today.",
  "Immediate action required: confirm banking details.",
  "Swiggy: your order has arrived.",
  "FREE antivirus scan available now.",
  "Quiz may be open notes according to the professor.",
  "Exclusive member-only discount ends tonight.",
  "Your order #18273 has shipped.",
  "Verify your identity to continue using your account.",
  "Meet me outside the metro station.",
  "Earn passive income from this secret trick.",
  "Bank OTP: 993182",
  "You have won a free iPhone.",
  "Hey, want to hit the gym later?",
  "Limited-time cashback available now.",
  "Google sign-in alert from a new device.",
  "Get rich fast with this one strategy.",
  "Assignment 4 rubric is now posted.",
  "Immediate payment required to avoid service interruption.",
  "Your package is at the front desk.",
  "Win guaranteed prizes today.",
  "Doctor says test reports are normal.",
  "Unlock your exclusive reward now.",
  "The interview got rescheduled to next week.",
  "Act now to claim your bonus.",
  "Reminder: electricity bill due tomorrow.",
  "URGENT reward claim notice.",
  "Thanks for helping with the project yesterday.",
  "Cheap meds available online without prescription.",
  "Train delayed by 20 mins.",
  "Exclusive flash sale ends in 2 hours.",
  "Can you grab coffee before class?",
  "Your Paytm transaction of ₹499 succeeded.",
  "Get approved instantly for a personal loan.",
  "Flight boarding starts in 25 minutes.",
  "Final reminder to claim your cashback.",
  "Let's play basketball this weekend.",
  "You have been shortlisted for a high-paying remote role.",
  "Seminar venue changed to Room 214.",
  "Confirm your account ownership immediately.",
  "Your food order was canceled and refunded.",
  "Lottery winner! Claim now.",
  "Dad reached safely.",
  "Immediate response required to avoid penalties.",
  "Library closes at 11 tonight.",
  "Amazing investment opportunity with guaranteed profit.",
  "Can you bring my notebook tomorrow?",
  "Click now to secure your free gift.",
  "Reminder: doctor follow-up this Friday.",
  "Suspicious payment attempt blocked.",
  "I left my hoodie in your car.",
  "Claim your welcome bonus today.",
  "Exam results are out.",
  "Get premium membership for free.",
  "The bus is 5 mins away.",
  "Urgent banking alert: verify transaction.",
  "Pizza tonight?",
  "Free giveaway ending soon."
]

    ground_truth = [
  "ham","spam","ham","spam","ham","spam","ham","spam","ham","spam",
  "ham","spam","ham","spam","ham","spam","ham","spam","ham","spam",
  "ham","spam","ham","spam","ham","spam","ham","spam","ham","spam",
  "ham","spam","ham","spam","ham","spam","ham","spam","spam","ham",
  "ham","spam","ham","spam","ham","spam","ham","spam","ham","spam",
  "ham","spam","ham","spam","ham","spam","ham","spam","ham","spam",
  "ham","spam","ham","spam","ham","spam","ham","spam","ham","spam",
  "ham","ham","spam","ham","spam","ham","spam","ham","spam","ham",
  "spam","ham","spam","ham","spam","ham","ham","spam","ham","ham",
  "spam","ham","spam","ham","spam","ham","spam"
]

    response = requests.post("http://localhost:8000/predict-multiple", 
    json={"messages": messages, "device_id": "threshold-test"})

    predictions = response.json()["Batch Predictions"]

    for threshold in [x/100 for x in range(50, 100)]:
        #True Positive, False Positive, False Negative, True Negative
        tp=fp=fn=tn=0
        for i, p in enumerate(predictions):
            predicted = "spam" if p["Classification"]=="spam" and p["Confidence"]>=threshold else "ham"
            actual = ground_truth[i]
            if predicted=="spam" and actual=="spam": tp+=1
            elif predicted=="spam" and actual=="ham": fp+=1
            elif predicted=="ham" and actual=="spam": fn+=1
            else: tn+=1

        #Calculating accuracy, precision, recall and f-1 off the 4 statistics gathered
        acc = (tp+tn)/(tp+fp+fn+tn)
        prec = tp/(tp+fp) if (tp+fp)>0 else 1
        rec = tp/(tp+fn) if (tp+fn)>0 else 0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
        print(f"threshold={threshold:.2f} acc={acc:.3f} prec={prec:.3f} rec={rec:.3f} f1={f1:.3f}")


#0.88 returned with the highest spam recall.

threshold=0.50 acc=0.763 prec=0.671 rec=1.000 f1=0.803
threshold=0.55 acc=0.784 prec=0.691 rec=1.000 f1=0.817
threshold=0.60 acc=0.794 prec=0.701 rec=1.000 f1=0.825
threshold=0.65 acc=0.794 prec=0.708 rec=0.979 f1=0.821
threshold=0.70 acc=0.804 prec=0.719 rec=0.979 f1=0.829
threshold=0.75 acc=0.814 prec=0.730 rec=0.979 f1=0.836
threshold=0.80 acc=0.825 prec=0.742 rec=0.979 f1=0.844
threshold=0.85 acc=0.835 prec=0.754 rec=0.979 f1=0.852
threshold=0.90 acc=0.856 prec=0.789 rec=0.957 f1=0.865
threshold=0.95 acc=0.835 prec=0.792 rec=0.894 f1=0.840
